# Getting started

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from nanotabpfn import (
    NanoTabPFNClassifier,
    NanoTabPFNModel,
    PriorDumpDataLoader,
    get_default_device,
    train,
)

In [ ]:
!test -f 300k_150x5_2.h5 || curl -L --skip-existing -o 300k_150x5_2.h5 "https://ndownloader.figshare.com/files/58932628?private_link=63fc1ada93e42e388e63"

In [ ]:
model = NanoTabPFNModel(
    embedding_size=96, num_attention_heads=4, mlp_hidden_size=192, num_layers=3, num_outputs=2
)

In [ ]:
prior = PriorDumpDataLoader(
    "300k_150x5_2.h5",
    num_steps=2500,
    batch_size=32,
)

In [ ]:
device = get_default_device()
device

In [ ]:
model, _ = train(model, prior, lr=4e-3, device=device)

In [ ]:
model

```
NanoTabPFNModel(
  (feature_encoder): FeatureEncoder(
    (linear_layer): Linear(in_features=1, out_features=96, bias=True)
  )
  (target_encoder): TargetEncoder(
    (linear_layer): Linear(in_features=1, out_features=96, bias=True)
  )
  (transformer_blocks): ModuleList(
    (0-2): 3 x TransformerEncoderLayer(
      (self_attention_between_datapoints): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
      )
      (self_attention_between_features): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
      )
      (linear1): Linear(in_features=96, out_features=192, bias=True)
      (linear2): Linear(in_features=192, out_features=96, bias=True)
      (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm3): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (decoder): Decoder(
    (linear1): Linear(in_features=96, out_features=192, bias=True)
    (linear2): Linear(in_features=192, out_features=2, bias=True)
  )
)
```

In [ ]:
clf = NanoTabPFNClassifier(model, device)

In [ ]:
X, y = load_breast_cancer(return_X_y=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5)

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_test_prob = clf.predict_proba(X_test)
y_test_pred = clf.predict(X_test)

In [ ]:
print(f"ROC AUC: {roc_auc_score(y_test, y_test_prob[:, 1]):_.2f}")
print(
    f"\nAccuracy: {accuracy_score(y_test, y_test_pred):_.0%}",
)

ROC AUC: 0.99

Accuracy: 95%